Experiment 1

In [3]:
%pip install pandas numpy pillow torch torchvision

  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 76.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 62.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 88.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 92.6 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 82.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 90.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 89.7 MB/s 

In [4]:
# imports
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset
from torchvision import transforms

In [5]:
# pathologies list
PATHOLOGIES = [
    'No Finding',
    'Enlarged Cardiomediastinum',
    'Cardiomegaly',
    'Lung Opacity',
    'Pneumonia',
    'Pleural Effusion',
    'Pleural Other',
    'Fracture',
    'Support Devices',
]

In [6]:
# function to pad images to a square
def pad_to_square(img):
    """Pad the shorter side with black pixels to make the image square."""
    w, h = img.size
    max_side = max(w, h)
    # (left, top, right, bottom)
    padding = (
        (max_side - w) // 2,
        (max_side - h) // 2,
        (max_side - w + 1) // 2,
        (max_side - h + 1) // 2,
    )
    return ImageOps.expand(img, padding, fill=0)

In [7]:
# image transform pipeline
TRANSFORM = transforms.Compose([
    transforms.Lambda(pad_to_square),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                         std=[0.229, 0.224, 0.225]),
])

In [11]:
# CheXpertDataset dataset for dataloader
class CheXpertDataset(Dataset):
    """
    A single dataset for one view type and one pathology.
    
    Args:
        df:         DataFrame (already filtered to frontal or lateral)
        pathology:  One of the 9 strings in PATHOLOGIES
        image_base: Root directory prepended to df['Path']
        transform:  Torchvision transform pipeline
    """
    def __init__(self, df, pathology, image_base, transform=TRANSFORM):
        assert pathology in PATHOLOGIES, f"Unknown pathology: {pathology}"
        self.image_base = image_base
        self.pathology = pathology
        self.transform = transform

        # Drop rows where this pathology label is NaN
        self.df = df[df[pathology].notna()].reset_index(drop=True)
        print(f"  [{pathology}] {len(self.df)} labeled rows")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_base, row['Path'])
        
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)

        # Label: 1.0 = positive, 0.0 = negative, -1.0 = uncertain (u-ones/u-zeros policy can replace later)
        label = torch.tensor(row[self.pathology], dtype=torch.float32)
        return img, label


In [9]:
# function to build datasets
# separates into frontal and lateral and within each angle separates into the 9 pathologies
# so we have 18
def build_datasets(csv_path, image_base):
    """
    Returns a nested dict:
        datasets['frontal']['Cardiomegaly'] -> CheXpertDataset
        datasets['lateral']['Pleural Effusion'] -> CheXpertDataset
        ...
    """
    df = pd.read_csv(csv_path)

    # Split on Frontal/Lateral column
    frontal_df = df[df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
    lateral_df = df[df['Frontal/Lateral'] == 'Lateral'].reset_index(drop=True)
    print(f"Frontal: {len(frontal_df)} rows | Lateral: {len(lateral_df)} rows\n")

    datasets = {'frontal': {}, 'lateral': {}}

    for view, view_df in [('frontal', frontal_df), ('lateral', lateral_df)]:
        print(f"--- {view.upper()} ---")
        for pathology in PATHOLOGIES:
            datasets[view][pathology] = CheXpertDataset(
                df=view_df,
                pathology=pathology,
                image_base=image_base,
            )

    return datasets


In [12]:
# main
datasets = build_datasets(
        csv_path='/resnick/groups/CS156b/from_central/data/student_labels/train2023.csv',
        image_base='/resnick/groups/CS156b/from_central/data/',
    )

# Access any sub-dataset
ds = datasets['frontal']['Cardiomegaly']
img, label = ds[0]
print(f"\nSample — image shape: {img.shape}, label: {label.item()}")

Frontal: 152298 rows | Lateral: 25860 rows

--- FRONTAL ---
  [No Finding] 152289 labeled rows
  [Enlarged Cardiomediastinum] 30174 labeled rows
  [Cardiomegaly] 33152 labeled rows
  [Lung Opacity] 83389 labeled rows
  [Pneumonia] 17873 labeled rows
  [Pleural Effusion] 89477 labeled rows
  [Pleural Other] 3783 labeled rows
  [Fracture] 8603 labeled rows
  [Support Devices] 92446 labeled rows
--- LATERAL ---
  [No Finding] 25860 labeled rows
  [Enlarged Cardiomediastinum] 8520 labeled rows
  [Cardiomegaly] 7756 labeled rows
  [Lung Opacity] 11375 labeled rows
  [Pneumonia] 4030 labeled rows
  [Pleural Effusion] 17233 labeled rows
  [Pleural Other] 1654 labeled rows
  [Fracture] 2045 labeled rows
  [Support Devices] 8054 labeled rows

Sample — image shape: torch.Size([3, 224, 224]), label: 0.0


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

class BasicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1: 3 -> 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)          # 224x224 -> 112x112
        )
        # Block 2: 32 -> 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)          # 112x112 -> 56x56
        )
        # Classifier head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # 56x56 -> 1x1 (avoids hardcoding spatial size)
            nn.Flatten(),
            nn.Linear(64, 1)         # single logit for binary classification
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x  # raw logit, BCEWithLogitsLoss handles sigmoid internally


In [14]:
def make_loss(dataset):
    """
    Compute pos_weight for BCEWithLogitsLoss from label distribution.
    pos_weight = num_negative / num_positive
    This tells the loss to penalize missing a positive more when positives are rare.
    """
    labels = dataset.df[dataset.pathology]
    n_pos = (labels == 1.0).sum()
    n_neg = (labels == 0.0).sum()
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
    print(f"  pos: {n_pos}, neg: {n_neg}, pos_weight: {pos_weight.item():.2f}")
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)  # (B,) -> (B, 1) to match output shape

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


In [16]:
# Example: train on frontal Cardiomegaly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ds = datasets['frontal']['Cardiomegaly']
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=4)

model = BasicCNN().to(device)
criterion = make_loss(ds).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_one_epoch(model, loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f}")

  pos: 17444, neg: 6985, pos_weight: 0.40


/groups/CS156b/from_central/2026/JDP/JDP-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/groups/CS156b/from_central/2026/JDP/JDP-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_wo

KeyboardInterrupt: 